In [1]:
from langchain_ollama import ChatOllama
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_community.document_loaders import PyPDFLoader
import pandas as pd
from langchain_chroma import Chroma
import torch
# print(torch.__version__, torch.version.cuda)
# print(torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())

C:\Users\ankit\AppData\Local\Temp\ipykernel_2628\1007429996.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


CUDA available: True


In [2]:
#importing dataset



# 1. Initialize the loader with your PDF file
loader = PyPDFLoader("S:/Projects/rag-dissertation/datasets/IPCC_AR6_SYR_FullVolume Climate.pdf")

# 2. Load and split the PDF into a list of Documents (one per page)
docs = loader.load()

# Or load and split into chunks directly
pages = loader.load_and_split()

print(f"Total pages loaded: {len(docs)}")
print(docs[0].page_content)  # Content of the first page
print(docs[0].metadata)  # Page number and source path

fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/BKSOIY+FrutigerLTPro-Bold', '/Encoding': IndirectObject(289, 0, 2179195928976), '/FirstChar': 2, '/FontDescriptor': IndirectObject(290, 0, 2179195928976), '/LastChar': 31, '/Subtype': '/Type1', '/Type': '/Font', '/Widths': [667, 278, 278, 556, 556, 611, 778, 556, 889, 611, 278, 333, 611, 611, 611, 611, 389, 389, 444, 556, 611, 889, 278, 278, 444, 389, 556, 556, 611, 722]}, but is not installed. Consider installing fontTools if you encounter encoding problems.
fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/BKQRXG+FrutigerLTPro-Condensed', '/Encoding': IndirectObject(293, 0, 2179195928976), '/FirstChar': 1, '/FontDescriptor': IndirectObject(294, 0, 2179195928976), '/LastChar': 173, '/Subtype': '/Type1', '/Type': '/Font', '/Widths': [426, 481, 296, 278, 444, 574, 259, 500, 481, 242, 481, 482, 481, 240, 482, 612, 685, 482, 426, 7

Total pages loaded: 186
1
CLIMATE CHANGE 2023
Synthesis Report
A Report of the Intergovernmental Panel on Climate Change
{'producer': 'iLovePDF', 'creator': 'Adobe InDesign 18.3 (Macintosh)', 'creationdate': '2023-07-15T11:35:23+09:00', 'moddate': '2023-08-15T19:45:58+09:00', 'trapped': '/False', 'source': 'S:/Projects/rag-dissertation/datasets/IPCC_AR6_SYR_FullVolume Climate.pdf', 'total_pages': 186, 'page': 0, 'page_label': 'i'}


In [ ]:
#loading dataset in langchain

# docs = []
# for col in df.columns:
#     if col == "Unnamed: 0":
#         continue
#     text = "\n".join(df[col].dropna().astype(str))
#     docs.append(Document(page_content=text, metadata={"column": col}))

# print(len(docs))   # = number of columns
# print(docs[0].metadata)
# print(type(docs[0].page_content))

1
{'column': 'deep_learning'}
<class 'str'>


In [3]:
#Spliting text

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
texts = text_splitter.split_documents(docs)
print(len(texts)) 
print(texts[5].page_content)

809
others of a similar nature, which are not mentioned or advertised. 
The right of publication in print, electronic and any other form and in any language is reserved by the IPCC. Short extracts 
from this publication may be reproduced without authorization provided that complete source is clearly indicated. Editorial 
correspondence and requests to publish, reproduce or translate articles in part or in whole should be addressed to: IPCC c/o World 
Meteorological Organization (WMO) 7bis, avenue de la Paix Tel.: +41 22 730 8208 P .O. Box 2300 Fax: +41 22 730 8025 CH 1211 
Geneva 2, Switzerland E-mail: IPCC-Sec@wmo.int www.ipcc.ch 
Paola Arias (Colombia), Mercedes Bustamante (Brazil), Ismail Elgizouli (Sudan), Gregory Flato (Canada), Mark Howden (Australia), 
Carlos Méndez (Venezuela), Joy Jacqueline Pereira (Malaysia), Ramón Pichs-Madruga (Cuba), Steven K Rose (USA), Yamina Saheb


In [2]:
#Embeddings Model 

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2",
    model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"},
    encode_kwargs={"batch_size": 512},
    )

embeddings._client.half()                 # fp16
embeddings._client.max_seq_length = 256   # truncate long abstracts

query_embedding = embeddings.embed_query("What is a sentence embedding?")
doc_embeddings = embeddings.embed_documents(
    [
        "Sentence embeddings map text to dense vectors.",
        "LangChain provides a standard Embeddings interface.",
    ]
)

print(type(embeddings))
print(hasattr(embeddings, "_client"), hasattr(embeddings, "client"))

s:\Projects\rag-dissertation\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 670.66it/s]


<class 'langchain_huggingface.embeddings.huggingface.HuggingFaceEmbeddings'>
True False


In [3]:
# Embeddings & vector storage
#
# This cell only READS the Chroma DB that was persisted to ./chroma_langchain_db
# (src/chroma_langchain_db) -- it does not re-embed the dataset on every run.

from pathlib import Path

CHROMA_DIR = Path("chroma_langchain_db")
if not CHROMA_DIR.exists():
    raise FileNotFoundError(
        f"Local Chroma DB not found at {CHROMA_DIR.resolve()}. "
        "Run the ingestion cells once with REBUILD_INDEX = True to create it."
    )

vector_store = Chroma(
    collection_name="Environment_Embeddings",
    embedding_function=embeddings,
    persist_directory=str(CHROMA_DIR),
)

# Default LangChain retriever over the persisted collection.
retriever = vector_store.as_retriever()

print(f"Loaded {vector_store._collection.count()} chunks from {CHROMA_DIR.resolve()}")


Loaded 809 chunks from S:\Projects\rag-dissertation\src\chroma_langchain_db


In [6]:
# Timeline of storage
#
# The index already exists on disk, so ingestion is opt-in: flip REBUILD_INDEX to
# True only when you want to (re)embed the split texts into the collection.

from tqdm import tqdm

REBUILD_INDEX = True

if REBUILD_INDEX:
    max_bs = vector_store._client.get_max_batch_size()   # 5461 for you
    BATCH = min(2000, max_bs)                            # stay safely under it

    for i in tqdm(range(0, len(texts), BATCH)):
        batch = texts[i:i + BATCH]
        ids = [str(d.metadata.get("pmid", i + j)) for j, d in enumerate(batch)]
        vector_store.add_documents(batch, ids=ids)
else:
    print("Skipping ingestion; reading the existing local Chroma DB as-is.")


100%|██████████| 1/1 [04:58<00:00, 298.46s/it]


In [4]:
# Retrieve evidence for the same question that will be evaluated.
query = "How is The probability of low-likelihood outcomes associated with very large impacts?"
retrieved_docs = retriever.invoke(query)
retrieval_context = [doc.page_content for doc in retrieved_docs]
if not retrieval_context:
    raise ValueError("The retriever returned no context; cannot evaluate a RAG answer.")
print(f"Retrieved {len(retrieval_context)} context chunks.")

# Show the retrieved chunks (rank, metadata, text) so the evidence is visible.
for rank, doc in enumerate(retrieved_docs, start=1):
    meta = ", ".join(f"{k}={v}" for k, v in doc.metadata.items()) or "none"
    print(f"--- chunk {rank}/{len(retrieved_docs)} [{meta}] ---")
    print(doc.page_content)
    print()


Retrieved 4 context chunks.
--- chunk 1/4 [creationdate=2023-07-15T11:35:23+09:00, page_label=126, creator=Adobe InDesign 18.3 (Macintosh), page=141, total_pages=186, producer=iLovePDF, trapped=/False, moddate=2023-08-15T19:45:58+09:00, source=S:/Projects/rag-dissertation/datasets/IPCC_AR6_SYR_FullVolume Climate.pdf] ---
political debate under the United Nations Framework Convention on 
Climate Change (UNFCCC) following the establishment of the Warsaw 
Mechanism on Loss and Damage in 2013, which is to ‘address loss 
and damage associated with impacts of climate change , including 
extreme events  and slow onset events, in developing countries  that 
are particularly vulnerable to the adverse effects of climate change.’ 
Lowercase letters ( losses and damages ) have been taken to refer 
broadly to harm from (observed) impacts and (projected) risks and can 
be economic or non-economic. 
Low-likelihood, high-impact outcomes 
Outcomes/events whose probability of occurrence is low or not we

In [5]:
#Setting up LLM

llm = ChatOllama(
    model="qwen3.5:2b", 
    temperature = 0.5,
    reasoning=False,
    
)

In [6]:
#Pre Retrieval - Baseline


response = llm.invoke(query)
print(response.content)

The probability of **low-likelihood outcomes** that are associated with **very large impacts** is the defining characteristic of **rare catastrophic events**. In risk management and decision theory, this relationship is often described as a **long-tail distribution**, specifically resembling an **inverse power law** or a **Pareto distribution**.

Here is a detailed breakdown of how this phenomenon works:

### 1. The Statistical Framework: The Long Tail
In most common distributions (like the Normal or Bell Curve), outcomes cluster around a central value with low probability for extreme values. However, in systems involving human error, technology failure, or systemic collapse, the distribution of outcomes often follows a **power law**:
$$P(x) \propto x^{-\alpha}$$
Where:
*   $x$ is the magnitude of the impact (or loss).
*   $\alpha$ is a positive constant.

In these distributions, while the probability of a tiny impact is high, the probability of a massive impact drops off rapidly at fi

In [7]:
#Pre Retrieval - Baseline Eval

import os

# The local judge can be slow, so allow longer local judge calls.
os.environ["DEEPEVAL_PER_ATTEMPT_TIMEOUT_SECONDS_OVERRIDE"] = "300"
os.environ["DEEPEVAL_PER_TASK_TIMEOUT_SECONDS_OVERRIDE"] = "600"

actual_context =[
    """The effectiveness of adaptation to reduce climate risk is documented
for specific contexts, sectors and regions and will decrease with
increasing warming (high confidence)125. For example, common
adaptation responses in agriculture - adopting improved cultivars and
agronomic practices, and changes in cropping patterns and crop
systems - will become less effective from 2°C to higher levels of
warming (high confidence). The effectiveness of most water-related
adaptation options to reduce projected risks declines with increasing
warming (high confidence). Adaptations for hydropower and
thermo-electric power generation are effective in most regions up to
1.5°C to 2°C, with decreasing effectiveness at higher levels of warming
(medium confidence). Ecosystem-based Adaptation is vulnerable to
climate change impacts, with effectiveness declining with increasing
global warming (high confidence). Globally, adaptation options related
to agroforestry and forestry have a sharp decline in effectiveness at 3°C,
with a substantial increase in residual risk (medium confidence).
{WGII SPM C.2, WGII SPM C.2.1, WGII SPM C.2.5, WGII SPM C.2.10,
WGII Figure TS.6 Panel (e), 4.7.2}"""
]

from deepeval.metrics import (
    AnswerRelevancyMetric,
    FaithfulnessMetric,
    HallucinationMetric,
)
from deepeval.models.llms.ollama_model import OllamaModel, retry_ollama
from deepeval.test_case import Audio, LLMTestCase
from pydantic import BaseModel
from typing import List, Optional, Tuple, Union


class NonThinkingOllamaModel(OllamaModel):
    @retry_ollama
    def generate(
        self,
        prompt: str,
        schema: Optional[BaseModel] = None,
        audio: Optional[List[Audio]] = None,
    ) -> Tuple[Union[str, BaseModel], float]:
        response = self.load_model().chat(
            model=self.name,
            messages=self._prompt_messages(prompt, audio),
            think=False,
            format=schema.model_json_schema() if schema else None,
            options={
                "temperature": self.temperature,
                **self.generation_kwargs,
            },
        )
        content = response.message.content
        if not content:
            raise ValueError(
                "Ollama returned empty content for the DeepEval judge response."
            )
        result = schema.model_validate_json(content) if schema else content
        return result, 0

# Use a separate local judge; Qwen 3.5 otherwise returns reasoning instead of JSON.
evaluation_model = NonThinkingOllamaModel(
    model="gemma4:e4b",
    base_url="http://localhost:11434",
    temperature=0,
    # reasoning=False,
)

test_case = LLMTestCase(
    input=query,
    actual_output=response.content,
    context=actual_context,
    retrieval_context=retrieval_context,
)

metrics = {
    "hallucination": HallucinationMetric(
        threshold=0.5, model=evaluation_model, include_reason=True, async_mode=False
    ),
    "faithfulness": FaithfulnessMetric(
        threshold=0.5, model=evaluation_model, include_reason=True, async_mode=False
    ),
    "answer relevancy": AnswerRelevancyMetric(
        threshold=0.5, model=evaluation_model, include_reason=True, async_mode=False
    ),
}

for name, metric in metrics.items():
    metric.measure(test_case)
    print(f"{name}: {metric.score:.3f}")
    print(f"  passed: {metric.is_successful()}")
    print(f"  reason: {metric.reason}")


s:\Projects\rag-dissertation\.venv\Lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets" for 
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

C:\Users\ankit\AppData\Local\Temp\ipykernel_2628\363103800.py:83: DeprecationWarning: 'HallucinationMetric' now scores in the same direction as every other deepeval metric: 1 is a pass, 0 is a failure, and 'threshold' is the MINIMUM passing score. It previously scored the proportion of violations, where 'threshold' was a maximum. Review any 'threshold' you pass and any code reading '.score' - a threshold of 0.2 that used to mean 'at most 20% violations' should now be 0.8. This notice will be removed in a future release.
  "hallucination": HallucinationMetric(


hallucination: 0.000
  passed: False
  reason: The score is 0.00 because the actual output discusses 'rare catastrophic events' and 'long-tail distribution' in risk management, which is completely unrelated to the provided context about the effectiveness of adaptation to reduce climate risk in agriculture, water-related options, and forestry.


faithfulness: 1.000
  passed: True
  reason: The score is 1.00 because there are no contradictions, meaning the actual output is perfectly faithful to the retrieval context.


answer relevancy: 0.846
  passed: True
  reason: The score is 0.85 because while the answer correctly addresses the core concept of low-likelihood, high-impact events, several irrelevant statements detract from a perfect score. Specifically, the output discusses probabilities of tiny impacts, contradictory probability patterns, and misrepresents the frequency/magnitude of the probability, which dilutes the focus on the primary relationship requested.


In [8]:
# Post Retrieval Query
context_text = "\n\n".join(retrieval_context)
prompt = f"""Answer the question using only the supplied context. If the context does not contain the answer, say so.

Context:
{context_text}

Question: {query}
Answer:"""
response = llm.invoke(prompt)
print(response.content)


Based on the context provided, outcomes classified as "Low-likelihood, high-impact" are those whose probability of occurrence is low or not well known (due to deep uncertainty), but whose potential impacts on society and ecosystems could be very large. Therefore, such outcomes are considered when they are associated with **very large consequences**.

Answer: Very large consequences


In [9]:
#Post Retrieval Eval
import os

# The local judge can be slow, so allow longer local judge calls.
os.environ["DEEPEVAL_PER_ATTEMPT_TIMEOUT_SECONDS_OVERRIDE"] = "300"
os.environ["DEEPEVAL_PER_TASK_TIMEOUT_SECONDS_OVERRIDE"] = "600"

actual_context =[
    """The effectiveness of adaptation to reduce climate risk is documented
    for specific contexts, sectors and regions and will decrease with
    increasing warming (high confidence)125. For example, common
    adaptation responses in agriculture - adopting improved cultivars and
    agronomic practices, and changes in cropping patterns and crop
    systems - will become less effective from 2°C to higher levels of
    warming (high confidence). The effectiveness of most water-related
    adaptation options to reduce projected risks declines with increasing
    warming (high confidence). Adaptations for hydropower and
    thermo-electric power generation are effective in most regions up to
    1.5°C to 2°C, with decreasing effectiveness at higher levels of warming
    (medium confidence). Ecosystem-based Adaptation is vulnerable to
    climate change impacts, with effectiveness declining with increasing
    global warming (high confidence). Globally, adaptation options related
    to agroforestry and forestry have a sharp decline in effectiveness at 3°C,
    with a substantial increase in residual risk (medium confidence).
    {WGII SPM C.2, WGII SPM C.2.1, WGII SPM C.2.5, WGII SPM C.2.10,
    WGII Figure TS.6 Panel (e), 4.7.2}"""
]

from deepeval.metrics import (
    AnswerRelevancyMetric,
    FaithfulnessMetric,
    HallucinationMetric,
)
from deepeval.models.llms.ollama_model import OllamaModel, retry_ollama
from deepeval.test_case import Audio, LLMTestCase
from pydantic import BaseModel
from typing import List, Optional, Tuple, Union


class NonThinkingOllamaModel(OllamaModel):
    @retry_ollama
    def generate(
        self,
        prompt: str,
        schema: Optional[BaseModel] = None,
        audio: Optional[List[Audio]] = None,
    ) -> Tuple[Union[str, BaseModel], float]:
        response = self.load_model().chat(
            model=self.name,
            messages=self._prompt_messages(prompt, audio),
            think=False,
            format=schema.model_json_schema() if schema else None,
            options={
                "temperature": self.temperature,
                **self.generation_kwargs,
            },
        )
        content = response.message.content
        if not content:
            raise ValueError(
                "Ollama returned empty content for the DeepEval judge response."
            )
        result = schema.model_validate_json(content) if schema else content
        return result, 0

# Use a separate local judge; Qwen 3.5 otherwise returns reasoning instead of JSON.
evaluation_model = NonThinkingOllamaModel(
    model="gemma4:e4b",
    base_url="http://localhost:11434",
    temperature=0,
    # reasoning=False,
)

test_case = LLMTestCase(
    input=query,
    actual_output=response.content,
    context=actual_context,
    retrieval_context=retrieval_context,
)

metrics = {
    "hallucination": HallucinationMetric(
        threshold=0.5, model=evaluation_model, include_reason=True, async_mode=False
    ),
    "faithfulness": FaithfulnessMetric(
        threshold=0.5, model=evaluation_model, include_reason=True, async_mode=False
    ),
    "answer relevancy": AnswerRelevancyMetric(
        threshold=0.5, model=evaluation_model, include_reason=True, async_mode=False
    ),
}

for name, metric in metrics.items():
    metric.measure(test_case)
    print(f"{name}: {metric.score:.3f}")
    print(f"  passed: {metric.is_successful()}")
    print(f"  reason: {metric.reason}")


s:\Projects\rag-dissertation\.venv\Lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets" for 
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

hallucination: 0.000
  passed: False
  reason: The score is 0.00 because the actual output introduces concepts ('Low-likelihood, high-impact' outcomes and 'very large consequences') that are entirely absent from the context, indicating a complete lack of factual grounding.


faithfulness: 1.000
  passed: True
  reason: The score is 1.00 because there are no contradictions, meaning the actual output is perfectly faithful to the retrieval context.


answer relevancy: 1.000
  passed: True
  reason: The score is 1.00 because the output directly and completely addresses the input question without any irrelevant statements, achieving the maximum possible relevance.
